<a href="https://colab.research.google.com/github/vee-16/contextual-deepfake-detection/blob/benchmarks/siglip_dinov2_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets pillow scikit-learn tqdm
!pip install --upgrade torchao
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from PIL import Image
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
!git clone https://huggingface.co/Bombek1/ai-image-detector-siglip-dinov2

import sys
sys.path.append("ai-image-detector-siglip-dinov2")

from huggingface_hub import hf_hub_download
from model import AIImageDetector

model_path = hf_hub_download(
    repo_id="Bombek1/ai-image-detector-siglip-dinov2",
    filename="pytorch_model.pt"
)

detector = AIImageDetector(model_path)

print("Bombek model loaded")

Cloning into 'ai-image-detector-siglip-dinov2'...
remote: Enumerating objects: 13, done.
remote: Total 13 (delta 0), reused 0 (delta 0), pack-reused 13 (from 1)
Receiving objects: 100% (13/13), 10.42 KiB | 10.42 MiB/s, done.
Resolving deltas: 100% (1/1), done.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


pytorch_model.pt:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Model loaded on cuda
Bombek model loaded


In [3]:
from datasets import load_dataset, Image as HFImage

dataset = load_dataset(
    "ComplexDataLab/OpenFake",
    split="test",
    streaming=True
)

# IMPORTANT: prevents Hugging Face from auto-opening images before our try/except
dataset = dataset.cast_column("image", HFImage(decode=False))

print(dataset)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/206 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/206 [00:00<?, ?it/s]

IterableDataset({
    features: ['image', 'prompt', 'label', 'model', 'type', 'release_date'],
    num_shards: 7
})


In [6]:
from PIL import Image, UnidentifiedImageError
import io
import numpy as np
from tqdm import tqdm

batch_size = 32

y_true = []
y_pred = []
y_prob = []

skipped = 0
batch_images = []
batch_labels = []

def run_single_image(image):
    result = detector.predict(image)

    prob_fake = float(result["probability"])

    if result["prediction"].lower() in ["ai", "fake", "deepfake", "generated"]:
        pred = 1
    else:
        pred = 0

    return pred, prob_fake

def flush_batch(batch_images, batch_labels):
    global y_true, y_pred, y_prob

    if len(batch_images) == 0:
        return

    for image, true_label in zip(batch_images, batch_labels):
        pred, prob_fake = run_single_image(image)

        y_true.append(true_label)
        y_pred.append(pred)
        y_prob.append(prob_fake)

for sample in tqdm(dataset):
    try:
        raw_image = sample["image"]

        if isinstance(raw_image, dict) and raw_image.get("bytes") is not None:
            image = Image.open(io.BytesIO(raw_image["bytes"])).convert("RGB")
        elif isinstance(raw_image, dict) and raw_image.get("path") is not None:
            image = Image.open(raw_image["path"]).convert("RGB")
        else:
            image = raw_image.convert("RGB")

        true = label_map[sample["label"]]

        batch_images.append(image)
        batch_labels.append(true)

        if len(batch_images) == batch_size:
            flush_batch(batch_images, batch_labels)
            batch_images = []
            batch_labels = []

    except (UnidentifiedImageError, OSError, KeyError, TypeError):
        skipped += 1
        continue

flush_batch(batch_images, batch_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print("Finished Bombek inference")
print("Total evaluated:", len(y_true))
print("Skipped images:", skipped)

770it [01:24,  6.13it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
8020it [13:36, 14.65it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
29459it [50:04, 12.17it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))
59658it [1:41:37,  9.78it/s]


Finished Bombek inference
Total evaluated: 59655
Skipped images: 3


In [5]:
label_map = {
    "real": 0,
    "fake": 1,
    "Real": 0,
    "Fake": 1,
    "Realism": 0,
    "Deepfake": 1
}

In [9]:
# Recompute predictions from probabilities
y_pred = (y_prob >= 0.5).astype(int)

print("Recomputed predictions using threshold 0.5")

Recomputed predictions using threshold 0.5


In [10]:
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc = roc_auc_score(y_true, y_prob)
cm = confusion_matrix(y_true, y_pred)

print("\n=== Bombek Model Metrics (Fixed) ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc:.4f}")
print("\nConfusion Matrix:")
print(cm)


=== Bombek Model Metrics (Fixed) ===
Accuracy : 0.9901
Precision: 0.9908
Recall   : 0.9894
F1-score : 0.9901
ROC-AUC  : 0.9994

Confusion Matrix:
[[29553   273]
 [  315 29514]]


In [11]:
import csv


csv_path = os.path.join( "siglip-dinov2_benchmark.csv")

cm_flat = cm.flatten()

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "accuracy", "precision", "recall", "f1_score", "roc_auc",
        "cm_00", "cm_01", "cm_10", "cm_11"
    ])
    writer.writerow([
        f"{acc:.6f}",
        f"{prec:.6f}",
        f"{rec:.6f}",
        f"{f1:.6f}",
        f"{roc:.6f}",
        cm_flat[0], cm_flat[1], cm_flat[2], cm_flat[3]
    ])

print("Saved metrics to:", csv_path)

Saved metrics to: siglip-dinov2_benchmark.csv
